# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. pd.series создает массив одномерный массив 10, 20, 30 затем параметр index присуждает ключи вместо обычных индексов на a b c затем принтом вызывается индекс ключ а iloc[] берет элемент массива игнорируя новые присужденные индексы итоговым ответом выйдет: (20 10)
2. создаются два одномерных массива a и b с одним одинаковым ключом Шымкент и двумя разными Астана и Алмата далее функция print() просит обьединить их и вывести на экран, но так как одинаковый ключ только один "Шымкент" выведет обьедененный результат обоих массивов только его, а у остальных выйдет NaN, также выйдет дататайп тут оно число с плавающей точкой окончательный ответ (
Алматы     NaN
Астана     NaN
Шымкент    3.0
dtype: float64)
3. pd.DataFrame() создает датафрейм с колонками x, y далее в print булевой маской фильтруем числа которые больше для колонки "x" 1 если больше то оставляем если меньше выбрасываем так первый элемент колонки x выбрасывается, но при этом вместе с ним элемент "4" также уходит вместе с "1" так как они в одном ряду из за этого размерность датафрейма меняется и метод shape показывает (2, 2)
4. код пытается вывести тип массива, первая команда с одинарными квадратными скобками будет выводить одинарынй массив то есть Series, двойные квадратные скобки выведут список колнок то есть 2Д э то DataFrame, аттрибут __name__ в это время убирает все лишние из название по типу название библиотеки класса и оставлет только название массива, итоговый ответ: (Series DataFrame)
5. переменная f отсылается к заданию номер три и также отсеивает все элементы, что меньше или равны 1 в колонке "x" таким же образом падает весь первый ряд и в массиве остается только два ряда, далее параметр index указывает на названия индексов а tolist переносит массив в обычный пайтоновский список из за его выходит [1, 2],также так как названия индексов в пандасе перманентны из за того что первый ряд отсеился названия индексов не поменяются на [0, 1], далее iloc() ищет элемент с координатами [0, 1] по уже отсееному датафрему, выйдет элемент "5" Итоговый результат: [1, 2] 5

In [2]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [4]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [5]:
# 1
print(f"Строк и столбцов {df.shape}")
print(f"Пропуски в delivery_days: {df["delivery_days"].isna().sum()}")

print(df[["city", "category", "status", "customer_id"]].nunique())

print("""1)В delivery_days имеется 160 пропусков что может повлечь ошибки в будущем, 
2)Имеется два статуса оплаченных заказов если не отфильтровать по только оплаченным можно получить ложную выручку
3)Нестандаризированные названия городов в столбце "city" одни и те же города записаны по
разному(разный регистр, смешание кирилицы и латиницы) по этому считаются отдельно и выходит 14 разных городов """ )

Строк и столбцов (5463, 8)
Пропуски в delivery_days: 160
city             14
category          5
status            2
customer_id    2003
dtype: int64
1)В delivery_days имеется 160 пропусков что может повлечь ошибки в будущем, 
2)Имеется два статуса оплаченных заказов если не отфильтровать по только оплаченным можно получить ложную выручку
3)Нестандаризированные названия городов в столбце "city" одни и те же города записаны по
разному(разный регистр, смешание кирилицы и латиницы) по этому считаются отдельно и выходит 14 разных городов 


## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [6]:
# 2
clean_df = df[(df["status"] =="оплачен") & (df["customer_id"] != "TEST")]
print(clean_df["amount"].quantile(0.25))
print(clean_df["amount"].quantile(0.50))
print(clean_df["amount"].quantile(0.75))
print(clean_df["amount"].mean())
print("Медиана показывает более честные средние заказы потому что не подвержена резким одиночным всплескам")

5880.0
10300.0
18690.0
14672.391348704346
Медиана показывает более честные средние заказы потому что не подвержена резким одиночным всплескам


## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [7]:
# 3

SH = ["Шымкент", "шымкент", "Shymkent"]
AL = ["Алматы", "алматы", "АЛМАТЫ", "Almaty", " Алматы"]

SH_df = df[df["city"].isin(SH)]
AL_df = df[df["city"].isin(AL)]

SH_share = (SH_df["status"] == "отменён").mean()
AL_share = (AL_df["status"] == "отменён").mean()
common_share = (df["status"] == "отменён").mean()

print(f"Доля отмененных заказов в Шымкенте: {SH_share: .2%}")
print(f"Доля отменных заказов в Алмате: {AL_share: .2%}")
print(f"Доля общих отменненых заказов: {common_share: .2%}")

print("""Доля отмененных заказов в Шымкенте больше в два раза от общей нормы отменненых заказов и почти на столько же от Алматы
такой результат сильно выделяется, потому что это значит что каждый пятый заказ оформленный из Шымкента был отменен""")


Доля отмененных заказов в Шымкенте:  20.40%
Доля отменных заказов в Алмате:  7.86%
Доля общих отменненых заказов:  10.23%
Доля отмененных заказов в Шымкенте больше в два раза от общей нормы отменненых заказов и почти на столько же от Алматы
такой результат сильно выделяется, потому что это значит что каждый пятый заказ оформленный из Шымкента был отменен


## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [8]:
# 4
is_sh = df["city"].isin(SH)
is_al = df["city"].isin(AL)

sh_sep = df.loc[sep & paid & not_test & is_sh, "amount"].sum()
sh_oct = df.loc[octo & paid & not_test & is_sh, "amount"].sum()

al_sep = df.loc[sep & paid & not_test & is_al, "amount"].sum()
al_oct = df.loc[octo & paid & not_test & is_al, "amount"].sum()

sh_change = (sh_oct - sh_sep) / sh_sep * 100
al_change = (al_oct - al_sep) / al_sep * 100

print(f"Шымкент: сентябрь = {sh_sep:.0f}, октябрь = {sh_oct:.0f}, изменение = {sh_change:.2f}%")
print(f"Алматы: сентябрь = {al_sep:.0f}, октябрь = {al_oct:.0f}, изменение = {al_change:.2f}%")
print(f"Во сколько раз падение в Шымкенте сильнее: {abs(sh_change) / abs(al_change):.2f}")

Шымкент: сентябрь = 4949800, октябрь = 4044010, изменение = -18.30%
Алматы: сентябрь = 15523940, октябрь = 14643020, изменение = -5.67%
Во сколько раз падение в Шымкенте сильнее: 3.22


## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
# 5
SH = ["Шымкент", "шымкент", "Shymkent"]
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))
print("В коде отсутствовали круглые скобки, а df[city] == 'Шымкент' учитывает только одно написание города и не охватывает все" )

## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [13]:
# 6
print(df["customer_id"].value_counts().head(5))

top_client = df["customer_id"].value_counts().index[0]
client_orders = df.loc[df["customer_id"] == top_client, ["order_id", "order_date", "city", "amount", "status"]]
print(client_orders)

print(f"Всего строк: {len(client_orders)}, уникальных order_id: {client_orders['order_id'].nunique()}")

customer_id
C01818    11
C01076     9
C01947     9
C01656     9
C00420     9
Name: count, dtype: int64
      order_id        order_date       city   amount   status
57      101125  2026-09-01 15:00    Шымкент   7350.0  оплачен
779     100096  2026-09-09 18:00    Шымкент  45270.0  оплачен
1371    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
1375    101504  2026-09-15 21:00     Алматы   8890.0  оплачен
2195    101860  2026-09-24 18:00     Астана   1490.0  оплачен
2617    101462  2026-09-29 09:00     Астана  22170.0  оплачен
2776    101495  2026-09-30 21:00     Алматы  20180.0  оплачен
3417    103416  2026-10-08 14:00  Караганда   4330.0  оплачен
4279    102644  2026-10-18 08:00     Астана  13600.0  оплачен
4610    104001  2026-10-21 19:00     Алматы  21990.0  оплачен
4778    104572  2026-10-23 18:00     Алматы  11140.0  оплачен
Всего строк: 11, уникальных order_id: 10


## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [11]:
# 7

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [12]:
# 8